# Chebyshev approximation for the nozzle inverse

Compare reciprocal fits on the original singular-value interval and prepare the exact and bounded approximate LCU probabilities with FlagQuantum.

## Approximation is a checked modeling choice before compilation

The visible loader qfunc prepares seven coefficient bits, loads a reduced
amplitude table into the upper slice and applies H on explicitly uniform
lower bits. FlagQuantum constructs this state preparation. The host chooses
how many low bits to approximate using a measured amplitude L2 tolerance;
this choice is not an unreported compiler precision downgrade.

Try comparing zero tolerance with the retained approximation. Amplitude L2
error, probability L2 error, reciprocal response error and actual RY resources
are distinct metrics. All 128 original LCU weights remain represented. A
smaller loader alone does not prove a cheaper complete linear solver or
sufficient solution accuracy; propagate algorithm edits to shared quantum.py
for the full comparison workflow.


Use the local FlagQuantum workflow.

In [ ]:
from flagquantum import synth
from flagquantum.synth import H
from flagquantum.synth import Output
from flagquantum.synth import QArray
from flagquantum.synth import QBit
from flagquantum.synth import allocate
from flagquantum.synth import inplace_prepare_amplitudes
from flagquantum.synth import qfunc
import math
import torch

from examples.synthesis.applications.cfd.qls_for_hybrid_solvers.chebyshev_approximation.overview.quantum import (
    SIZE,
    WIDTH,
    approximation,
)


@qfunc
def load_distribution(
    amplitudes: list[float], uniform_bits: int, block: Output[QArray[QBit]]
) -> None:
    allocate(WIDTH, block)
    inplace_prepare_amplitudes(amplitudes, 0.0, block[uniform_bits:WIDTH])
    for bit in range(uniform_bits):
        H(block[bit])


def compile_preview(
    probabilities: torch.Tensor, tolerance: float
) -> tuple[synth.Program, int]:
    if probabilities.shape != (SIZE,) or bool((probabilities < 0).any()):
        raise ValueError("Provide the 128 nonnegative LCU weights")
    if not math.isclose(float(probabilities.sum()), 1.0, rel_tol=0, abs_tol=1e-12):
        raise ValueError("The LCU probabilities must sum to one")
    bits, reduced = approximation(probabilities, tolerance)
    program = synth.synthesize(
        load_distribution,
        classical_inputs={
            "amplitudes": list(map(float, reduced.sqrt())),
            "uniform_bits": bits,
        },
        max_qubits=WIDTH,
        max_workspace=0,
        max_operations=512,
        memory_limit_bytes=4 * 1024**2,
    )
    return program, bits


from pathlib import Path

from IPython.display import Image, display

from examples.synthesis.applications.cfd.qls_for_hybrid_solvers.chebyshev_approximation.overview import (
    model,
    quantum,
    workflow,
)
from examples.synthesis.applications.cfd.qls_for_hybrid_solvers.chebyshev_approximation.overview.main import (
    print_report,
)
from examples.synthesis.applications.cfd.qls_for_hybrid_solvers.chebyshev_approximation.overview.plots import (
    save_figures,
)


Choose the output directory for the plots.

In [ ]:
figures = Path("notebook-figures")


Compare both error metrics on the original singular-value interval.

In [ ]:
print(model.APPROXIMATIONS)


Restore the full original nozzle matrix and compute the banded scaling.

In [ ]:
matrix = model.source_matrix()
nozzle = model.build_model()


Inspect the dense matrix used for the singular-value calculation.

In [ ]:
print("Matrix shape:", tuple(matrix.shape))
print("Nonzero entries:", int(matrix.count_nonzero()))


Measure the scaled singular values and keep the original polynomial degrees.

In [ ]:
print("Banded normalization:", nozzle.alpha)
print("Singular value interval:", nozzle.smallest, nozzle.largest)


Generate the four polynomials and compute their errors.

In [ ]:
cases = workflow.approximations(nozzle)
for case in cases:
    print(case.method, case.degree, case.relative_error, case.uniform_error)


Fit all 128 inverse LCU coefficients, then prepare both distributions.

In [ ]:
fitted = model.fit_alternating(cases[1].coefficients)
exact, approximate = quantum.compare(fitted)
result = workflow.Result(nozzle, cases, fitted, exact, approximate)
print_report(result)


Check that the approximate circuit uses less than half as many RY gates.

In [ ]:
assert (
    approximate.resources.operation_counts["ry"] * 2
    < exact.resources.operation_counts["ry"]
)


Plot the reciprocal approximations and measured coefficient distributions.

In [ ]:
save_figures(result, figures)
display(Image(filename=str(figures / "reciprocal-approximations.png")))
display(Image(filename=str(figures / "prepared-coefficients.png")))


preview, uniform_bits = compile_preview(fitted.abs() / fitted.abs().sum(), 0.03)
print("Explicit uniform low bits:", uniform_bits)
print("Editable loader resources:", preview.report["resources"])
